# Setup environment (one-time, by hand)
- Creates the catalog, schemas and volumes for one environment of this company.
- Bronze volumes are **external** on S3 (`landing`, `processed`, `rejected`); ops volumes are **managed**.
- Pick **Environment**, run with **Mode = preview**, check the SQL, then set **Mode = apply** and Run all.
- Needs the environment's external location `lh_<company>_<env>_source` to exist first. Not part of any job.

## Repo constant and widgets

In [0]:
# The ONLY line that differs between repos (fqf repo: "fqf")
REPO_COMPANY = "bhg"
COMPANY_NAME = "Bio Health Group"

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"], "Environment")
dbutils.widgets.dropdown("mode", "preview", ["preview", "apply"], "Mode")

## Configuration (all names derived; nothing typed by hand)

In [0]:
env  = dbutils.widgets.get("env")
mode = dbutils.widgets.get("mode")

catalog = f"{REPO_COMPANY}_{env}"                 # bhg_dev
bucket  = f"lh-{REPO_COMPANY}-{env}-source"       # lh-bhg-dev-source
ext_loc = f"lh_{REPO_COMPANY}_{env}_source"       # lh_bhg_dev_source

SCHEMAS = {
    "bronze": "Raw data, as landed",
    "silver": "Cleaned and conformed data",
    "gold":   "Business-ready tables",
    "ops":    "Audit, watermarks and checks",
}
BRONZE_VOLUMES = {                                 # external, on S3
    "landing":   "New source files: <source>/<table>/yyyy/mm/",
    "processed": "Files loaded into bronze successfully",
    "rejected":  "Files that failed checks",
}
OPS_VOLUMES = {"checkpoints": "Auto Loader progress"}   # managed
if env in ("dev", "test"):
    OPS_VOLUMES["refresh_in"] = "Masked refresh files received"
if env in ("test", "prod"):
    OPS_VOLUMES["refresh_out"] = "Masked refresh files waiting to be sent"

print(f"company name = {COMPANY_NAME} \ncatalog = {catalog} \nbucket = {bucket} \nexternal_location = {ext_loc} \nmode = {mode}")

## Pre-check that the external location exists and points at the right bucket

In [0]:
try:
    loc = spark.sql(f"DESCRIBE EXTERNAL LOCATION `{ext_loc}`").collect()[0].asDict()
except Exception as e:
    raise Exception(f"External location '{ext_loc}' not found. Do connection steps 1-4 for {catalog} first.") from e

expected_url = f"s3://{bucket}"
if loc["url"].rstrip("/") != expected_url:
    raise Exception(f"'{ext_loc}' points to {loc['url']}, expected {expected_url}/")
print(f"OK: {ext_loc} -> {loc['url']} (credential {loc['credential_name']})")

## Build the SQL

In [0]:
stmts = [f"CREATE CATALOG IF NOT EXISTS {catalog} COMMENT '{COMPANY_NAME}, {env} environment'"]
stmts += [f"CREATE SCHEMA IF NOT EXISTS {catalog}.{s} COMMENT '{c}'" for s, c in SCHEMAS.items()]
stmts += [f"CREATE EXTERNAL VOLUME IF NOT EXISTS {catalog}.bronze.{v} LOCATION 's3://{bucket}/{v}/' COMMENT '{c}'"
          for v, c in BRONZE_VOLUMES.items()]
stmts += [f"CREATE VOLUME IF NOT EXISTS {catalog}.ops.{v} COMMENT '{c}'" for v, c in OPS_VOLUMES.items()]

for s in stmts:
    print(s + ";")

## Apply (only when Mode = apply)

In [0]:
if mode == "apply":
    for s in stmts:
        spark.sql(s)
    print(f"Applied {len(stmts)} statements to {catalog}")
else:
    print("Preview only. Set Mode = apply and Run all to create.")

## Verify, and flag anything missing or extra

In [0]:
df = spark.sql(f"""
  SELECT volume_schema, volume_name, volume_type, storage_location
  FROM system.information_schema.volumes
  WHERE volume_catalog = '{catalog}'
  ORDER BY volume_schema, volume_name""")
display(df)

expected = {("bronze", v) for v in BRONZE_VOLUMES} | {("ops", v) for v in OPS_VOLUMES}
actual   = {(r.volume_schema, r.volume_name) for r in df.collect()}
print("Missing:", sorted(expected - actual) or "none")
print("Extra (overhead):", sorted(actual - expected) or "none")

## Schemas 

In [0]:
if spark.sql(f"SHOW CATALOGS LIKE '{catalog}'").count() == 1:
    display(spark.sql(f"SHOW SCHEMAS IN {catalog}"))
else:
    print(f"Catalog {catalog} does not exist yet (run with Mode = apply).")